# 🔓 Solutions: Model Training

> **Try first, peek second.** You learn far more by struggling with a problem for ten minutes than by reading its answer for ten seconds. Use these solutions to *compare* your approach, not to replace it. There is often more than one correct way.


In [1]:
import sys
sys.path.append("..")
from course_utils import *
setup()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score

def rmse(y_true, y_pred):
    """Root mean squared error: the typical size of a prediction error, in the target's own units."""
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

### 🧩 Problem 4.1: The classic 80 / 20 split &nbsp; ⭐

Split `students.csv` into train and test sets. The target is `exam_score`; all other columns are features.

Use `test_size=0.2` and `random_state=42`, and name the four pieces `X_train, X_test, y_train, y_test`.

In [2]:
students = load("students.csv")
X = students.drop(columns="exam_score")
y = students["exam_score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(160, 4) (40, 4)


In [3]:
# ✔ Self-check
check("160 training rows", lambda: len(X_train) == 160 and len(y_train) == 160)
check("40 test rows", lambda: len(X_test) == 40 and len(y_test) == 40)
check("no row appears in both sets", lambda: len(set(X_train.index) & set(X_test.index)) == 0)
check("features and target stay aligned", lambda: (X_train.index == y_train.index).all())

✅ PASSED   160 training rows
✅ PASSED   40 test rows
✅ PASSED   no row appears in both sets
✅ PASSED   features and target stay aligned


True

**🧑‍🏫 Walkthrough.** `test_size=0.2` puts 20% (40 of 200) into the test set. The check about indices matters: features and target are split *together*, row by row, so each house keeps its own price. Never shuffle `X` and `y` separately!

### 🧩 Problem 4.2: Stratify the churners &nbsp; ⭐⭐

Split `churn.csv` (target: `churned`) into train and test sets with `test_size=0.25`, `random_state=1`, **stratified on the target**.

Name them `X_train, X_test, y_train, y_test` and also compute the churn rate in each set: `rate_train`, `rate_test`.

In [4]:
churn = load("churn.csv")
X = churn.drop(columns="churned")
y = churn["churned"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)
rate_train, rate_test = y_train.mean(), y_test.mean()
print(f"train {rate_train:.4f}   test {rate_test:.4f}   overall {y.mean():.4f}")

train 0.1520   test 0.1520   overall 0.1520


In [5]:
# ✔ Self-check
check("test set is 25% of the data", lambda: len(X_test) == 375)
check("churn rates in train and test differ by < 0.2 percentage points", lambda: abs(rate_train - rate_test) < 0.002)
check("both rates match the overall rate", lambda: abs(rate_train - y.mean()) < 0.003)

✅ PASSED   test set is 25% of the data
✅ PASSED   churn rates in train and test differ by < 0.2 percentage points
✅ PASSED   both rates match the overall rate


True

**🧑‍🏫 Walkthrough.** Stratification guarantees that each split mirrors the class balance of the full dataset. Without it, a rare class could be under- or over-represented purely by chance, making the evaluation noisy and sometimes unusable (a test set with *no* positives can't even compute recall!).

### 🧩 Problem 4.3: Three-way split &nbsp; ⭐⭐

Split `housing.csv` (target: `price_k`) into **train 60% / validation 20% / test 20%** with two calls to `train_test_split`, using `random_state=0` both times.

Name the pieces `X_train, X_val, X_test, y_train, y_val, y_test`.

In [6]:
housing = load("housing.csv")
X = housing.drop(columns="price_k")
y = housing["price_k"]

X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.20, random_state=0)
X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=0.25, random_state=0)
print(len(X_train), len(X_val), len(X_test))

540 180 180


In [7]:
# ✔ Self-check
check("train has 540 rows (60%)", lambda: len(X_train) == 540)
check("validation has 180 rows (20%)", lambda: len(X_val) == 180)
check("test has 180 rows (20%)", lambda: len(X_test) == 180)
check("the three sets are disjoint", lambda: len(set(X_train.index) | set(X_val.index) | set(X_test.index)) == 900)

✅ PASSED   train has 540 rows (60%)
✅ PASSED   validation has 180 rows (20%)
✅ PASSED   test has 180 rows (20%)
✅ PASSED   the three sets are disjoint


True

**🧑‍🏫 Walkthrough.** The test set is cut off first and then never touched again. The second split uses 0.25 because it applies to the remaining 720 rows: 0.25 × 720 = 180. The disjointness check confirms the 540 + 180 + 180 rows are all distinct houses.

### 🧩 Problem 4.4: Choose k with validation, report with test &nbsp; ⭐⭐

Below we give you a ready-made three-way split of the **housing** data. Your job:

1. For each `k` in `[1, 3, 5, 10, 20, 50]` train `make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k))` on the **training** set and compute the **validation RMSE**. Store them in `val_rmse` (`{k: rmse}`).
2. Store the winning *k* in `best_k`, and its fitted pipeline in `best_model`.
3. Compute the **test RMSE** of `best_model` and store it in `test_rmse`.

*(Use the `rmse(y_true, y_pred)` helper defined at the top of this notebook.)*

In [8]:
housing = load("housing.csv")
X = housing.drop(columns="price_k"); y = housing["price_k"]
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.20, random_state=0)
X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=0.25, random_state=0)

val_rmse, fitted = {}, {}
for k in [1, 3, 5, 10, 20, 50]:
    fitted[k] = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k)).fit(X_train, y_train)
    val_rmse[k] = rmse(y_val, fitted[k].predict(X_val))

best_k = min(val_rmse, key=val_rmse.get)
best_model = fitted[best_k]
test_rmse = rmse(y_test, best_model.predict(X_test))      # first and only look at the test set
print({k: round(v, 1) for k, v in val_rmse.items()})
print(f"best k = {best_k};  validation RMSE = {val_rmse[best_k]:.1f};  TEST RMSE = {test_rmse:.1f}")

{1: 83.0, 3: 68.5, 5: 63.3, 10: 66.9, 20: 69.1, 50: 75.8}
best k = 5;  validation RMSE = 63.3;  TEST RMSE = 70.8


In [9]:
# ✔ Self-check
check("validation RMSE recorded for 6 values of k", lambda: sorted(val_rmse) == [1, 3, 5, 10, 20, 50])
check("best_k really has the lowest validation RMSE", lambda: best_k == min(val_rmse, key=val_rmse.get))
check("test RMSE matches best_model", lambda: abs(test_rmse - rmse(y_test, best_model.predict(X_test))) < 1e-9)
check("test RMSE is in the same ballpark as validation RMSE", lambda: 0.5 * val_rmse[best_k] < test_rmse < 2 * val_rmse[best_k])

✅ PASSED   validation RMSE recorded for 6 values of k
✅ PASSED   best_k really has the lowest validation RMSE
✅ PASSED   test RMSE matches best_model
✅ PASSED   test RMSE is in the same ballpark as validation RMSE


True

**🧑‍🏫 Walkthrough.** The workflow is: **loop → validate → choose → test once**. The test RMSE is usually a bit *worse* than the validation RMSE of the winner, because choosing the best of several candidates on the validation set is slightly optimistic (a mild "winner's curse"). If your test number were *wildly* different from validation, suspect a small split or a data problem.

### 🧩 Problem 4.5: Catch the leak &nbsp; ⭐⭐

Detective time. The DataFrame `leaky_df` below is the housing data plus two new columns: `listing_id` (a random ID) and `monthly_mortgage_usd`. One of the columns **leaks** the target.

1. Find the leaky column and store its name in `leaky_col` (hint: look at each feature's **correlation with `price_k`**, *and* ask whether it would be known before the sale).
2. Using `train_test_split(..., test_size=0.2, random_state=0)` and a `LinearRegression`, compute the test **R²** *with* all features → `leaky_r2`, and *without* the leaky column → `honest_r2`.

In [10]:
rng = np.random.default_rng(0)
leaky_df = load("housing.csv")
leaky_df["listing_id"] = rng.integers(10_000, 99_999, len(leaky_df))
leaky_df["monthly_mortgage_usd"] = (leaky_df["price_k"] * 1000 * 0.0056 * (1 + rng.normal(0, 0.01, len(leaky_df)))).round(0)

print(leaky_df.corr()["price_k"].abs().sort_values(ascending=False).round(3).head(4))
leaky_col = "monthly_mortgage_usd"

def test_r2(df):
    Xa, ya = df.drop(columns="price_k"), df["price_k"]
    Xtr, Xte, ytr, yte = train_test_split(Xa, ya, test_size=0.2, random_state=0)
    return r2_score(yte, LinearRegression().fit(Xtr, ytr).predict(Xte))

leaky_r2 = test_r2(leaky_df)
honest_r2 = test_r2(leaky_df.drop(columns=leaky_col))
print(f"leaky R² = {leaky_r2:.4f}   honest R² = {honest_r2:.4f}")

price_k                 1.000
monthly_mortgage_usd    1.000
area_sqm                0.889
bedrooms                0.728
Name: price_k, dtype: float64
leaky R² = 0.9993   honest R² = 0.9713


In [11]:
# ✔ Self-check
check("identified the leaky column", lambda: leaky_col == "monthly_mortgage_usd")
check("leaky model looks 'too good' (R² > 0.995)", lambda: leaky_r2 > 0.995)
check("honest model is lower but still good (0.90 < R² < 0.995)", lambda: 0.90 < honest_r2 < 0.995)

✅ PASSED   identified the leaky column
✅ PASSED   leaky model looks 'too good' (R² > 0.995)
✅ PASSED   honest model is lower but still good (0.90 < R² < 0.995)


True

**🧑‍🏫 Walkthrough.** The mortgage payment correlates ≈ 1.00 with price because it *is* price in disguise, and it can't exist before the price is agreed. `listing_id` is harmless noise (correlation ≈ 0). Correlation helps you *spot* suspects, but only **reasoning about when each feature becomes known** convicts them: a high correlation alone might just be a genuinely great feature (like area).

### 🧩 Problem 4.6: Write your own training loop &nbsp; ⭐⭐⭐

Reproduce the learning-curve experiment yourself. The data is prepared for you (scaled student features, with a train/validation split).

1. Create `SGDRegressor(learning_rate="constant", eta0=0.01, penalty=None, random_state=0)`.
2. Loop for **30 epochs**: call `partial_fit(X_tr, y_tr)` once per epoch, then append the **validation RMSE** to the list `val_curve`.
3. Store the epoch (1-based) at which the validation error was lowest in `best_epoch`.

In [12]:
students = load("students.csv")
Xs, ys = students.drop(columns="exam_score"), students["exam_score"]
Xa, Xb, y_tr, y_va = train_test_split(Xs, ys, test_size=0.2, random_state=0)
sc = StandardScaler().fit(Xa)
X_tr, X_va = sc.transform(Xa), sc.transform(Xb)

model = SGDRegressor(learning_rate="constant", eta0=0.01, penalty=None, random_state=0)
val_curve = []
for epoch in range(30):
    model.partial_fit(X_tr, y_tr)                       # one epoch
    val_curve.append(rmse(y_va, model.predict(X_va)))   # log the validation error

best_epoch = int(np.argmin(val_curve)) + 1
print(f"best epoch = {best_epoch};  RMSE at epoch 1 = {val_curve[0]:.2f}, at epoch 30 = {val_curve[-1]:.2f}")

best epoch = 7;  RMSE at epoch 1 = 11.86, at epoch 30 = 4.47


In [13]:
# ✔ Self-check
check("30 epochs recorded", lambda: len(val_curve) == 30)
check("error decreased overall (epoch 30 beats epoch 1)", lambda: val_curve[-1] < val_curve[0])
check("converged to a small error (< 5.5 points)", lambda: val_curve[-1] < 5.5)
check("best_epoch is the argmin (1-based)", lambda: best_epoch == int(np.argmin(val_curve)) + 1)

✅ PASSED   30 epochs recorded
✅ PASSED   error decreased overall (epoch 30 beats epoch 1)
✅ PASSED   converged to a small error (< 5.5 points)
✅ PASSED   best_epoch is the argmin (1-based)


True

**🧑‍🏫 Walkthrough.** This is the skeleton of *every* training loop in machine learning, from this tiny linear model to a billion-parameter neural net: *update, measure, log, repeat.* The "best epoch" may be a few epochs in with a tiny wiggle afterwards: once the curve is flat, differences are just noise. In deep learning, you'd stop training at the best epoch (**early stopping**).